BIOI-TAGGING

In [1]:
import json
from tqdm import tqdm
import spacy

In [2]:
nlp = spacy.blank("en")

# BIO-TAGGING:
def get_bio_tags(review_text, annotations):
    doc = nlp(review_text)
    tokens = [t.text for t in doc]
    tags = ["O"] * len(tokens)

    for ann in annotations:
        aspect = ann['aspect'].lower()
        sentiment = ann['sentiment']

        label = "POS" if sentiment == 1 else "NEG" if sentiment == -1 else "NEU"

        aspect_tokens = [t.text for t in nlp(aspect.lower())]
        n_aspect = len(aspect_tokens)

        for i in range(len(tokens) - n_aspect + 1):
            if [t.lower() for t in tokens[i:i+n_aspect]] == aspect_tokens:
                tags[i] = f"B-{label}"
                for j in range(1, n_aspect):
                    tags[i+j] = f"I-{label}"
                break
    return tokens, tags

In [3]:
review_samples = []
with open('absa_checkpoints.jsonl', 'r') as f:
    for line in tqdm(f, desc="Converting to BIO"):
        item = json.loads(line)
        tokens, tags = get_bio_tags(item['review'], item['annotations'])
        review_samples.append((tokens, tags))
review_samples[0]

Converting to BIO: 5248it [00:02, 1945.86it/s]


(['put',
  'this',
  'movie',
  'out',
  'of',
  'its',
  'misery',
  'and',
  'burn',
  'the',
  'negatives',
  'what',
  'am',
  'i',
  'saying',
  'the',
  'whole',
  'movie',
  'was',
  'negative',
  'fortunately',
  'only',
  'a',
  'very',
  'few',
  'would',
  'find',
  'this',
  'movie',
  'the',
  'least',
  'bit',
  'appealing',
  'this',
  'is',
  'what',
  'the',
  'vast',
  'american',
  'majority',
  'would',
  'call',
  'too',
  'much',
  'sex',
  'and',
  'violence',
  'it',
  'will',
  'probably',
  'show',
  'up',
  'on',
  'some',
  'nonpremium',
  'cable',
  'channel',
  'someday',
  'just',
  'for',
  'the',
  'shock',
  'value',
  'but',
  'after',
  'editing',
  'out',
  'the',
  'nudity',
  'most',
  'of',
  'the',
  'violence',
  'will',
  'stay',
  'all',
  'that',
  'will',
  'be',
  'left',
  'is',
  'minutes',
  'of',
  'really',
  'bad',
  'acting',
  'interspersed',
  'with',
  'minutes',
  'of',
  'commercials',
  'there',
  'are',
  'just',
  'too',
  '

In [8]:
from gensim.models import Word2Vec
import torch
import torch.nn as nn
import numpy as np

In [9]:
model_w2v = Word2Vec.load('hinglish_absa_w2v.model')
embedding_dim = model_w2v.vector_size
embedding_dim

300

GENERATING WEIGHT MATRIX

In [10]:
# Word2Index mapping
word2idx = {"<PAD>": 0, "<UNK>": 1}
for word in model_w2v.wv.index_to_key:
    word2idx[word] = len(word2idx)

# Weight matrix:
vocab_size = len(word2idx)
weight_matrix = np.zeros((vocab_size,embedding_dim))

for word, idx in word2idx.items():
  if word in model_w2v.wv:
    weight_matrix[idx] = model_w2v.wv[word]
  elif word == "<UNK>":
    weight_matrix[idx] = np.random.normal(scale=0.6, size=(embedding_dim,))

weight_matrix = torch.FloatTensor(weight_matrix)

tag2idx = {
    "O": 0,
    "B-POS": 1, "I-POS": 2,
    "B-NEG": 3, "I-NEG": 4,
    "B-NEU": 5, "I-NEU": 6
}
num_tags = len(tag2idx)

In [8]:
!pip install pytorch-crf

In [11]:
from torchcrf import CRF

In [45]:
class Bi_LSTM_CRF(nn.Module):
  def __init__(self,vocab_size,num_tags,embedding_dim,hidden_dim,weight_matrix):
    super(Bi_LSTM_CRF,self).__init__()

    self.embedding = nn.Embedding.from_pretrained(weight_matrix, freeze=True)
    self.lstm = nn.LSTM(embedding_dim, hidden_dim//2, num_layers=1, bidirectional=True, batch_first=True)
    self.dropout = nn.Dropout(0.5)
    self.hidden2tag = nn.Linear(hidden_dim, num_tags)
    self.crf = CRF(num_tags, batch_first = True)

  def forward(self, x, tags=None, mask= None):
    embds = self.embedding(x)
    lstm_out,_ = self.lstm(embds)
    emissions = self.hidden2tag(self.dropout(lstm_out))

    if tags is not None:
      return -self.crf(emissions, tags, mask=mask)
    else:
      return self.crf.decode(emissions)
hidden_dim = 128
model = Bi_LSTM_CRF(vocab_size, num_tags, embedding_dim, hidden_dim, weight_matrix)

In [46]:
model

Bi_LSTM_CRF(
  (embedding): Embedding(27572, 300)
  (lstm): LSTM(300, 64, batch_first=True, bidirectional=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (hidden2tag): Linear(in_features=128, out_features=7, bias=True)
  (crf): CRF(num_tags=7)
)

In [24]:
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

DATA PIPELINE

In [47]:
class ABSA_dataset(Dataset):
  def __init__(self,review_samples,word2idx,tag2idx):
    self.data = review_samples
    self.word2idx = word2idx
    self.tag2idx = tag2idx
  def __len__(self):
    return len(self.data)

  def __getitem__(self,idx):
    tokens, tags = self.data[idx]
    word_indices = [self.word2idx.get(token.lower(), self.word2idx['<UNK>']) for token in tokens]
    tags_indices = [self.tag2idx[tag] for tag in tags]

    return torch.LongTensor(word_indices), torch.LongTensor(tags_indices)

def collate_fn(batch):
    # This function handles the padding for each batch dynamically
    sentences, tags = zip(*batch)
    sentences_padded = pad_sequence(sentences, batch_first=True, padding_value=0)
    tags_padded = pad_sequence(tags, batch_first=True, padding_value=0)
    mask = (sentences_padded != 0).to(torch.uint8)

    return sentences_padded, tags_padded, mask

TRAIN_TEST_SPLIT

In [48]:
from sklearn.model_selection import train_test_split
train_data,test_data = train_test_split(review_samples,test_size=0.2, random_state=42)
print(f"Training on {len(train_data)} reviews")
print(f"Testing on {len(test_data)} reviews")

Training on 4198 reviews
Testing on 1050 reviews


In [49]:
train_data = ABSA_dataset(train_data,word2idx,tag2idx)
test_data = ABSA_dataset(test_data,word2idx,tag2idx)

train_loader = DataLoader(train_data, batch_size=32, shuffle= True, collate_fn = collate_fn)
test_loader = DataLoader(test_data, batch_size=32, shuffle= True, collate_fn = collate_fn)

TRAINING / TESTING

In [50]:
import torch.optim as optim
from seqeval.metrics import classification_report,f1_score

In [51]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)

best_test_loss = float('inf')

for epoch in range(20):
    model.train()
    total_train_loss = 0
    for tokens, tags, masks in train_loader:
        tokens, tags, masks = tokens.to(device), tags.to(device), masks.to(device)
        optimizer.zero_grad()
        loss = model(tokens, tags=tags, mask=masks)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        total_train_loss += loss.item()

    #Validation
    model.eval()
    total_test_loss = 0
    with torch.no_grad():
        for tokens, tags, masks in test_loader:
            tokens, tags, masks = tokens.to(device), tags.to(device), masks.to(device)
            loss = model(tokens, tags=tags, mask=masks)
            total_test_loss += loss.item()

    avg_test_loss = total_test_loss / len(test_loader)
    print(f"Epoch {epoch+1} | Train Loss: {total_train_loss/len(train_loader):.4f} | Test Loss: {avg_test_loss:.4f}")


    if avg_test_loss < best_test_loss:
        best_test_loss = avg_test_loss
        torch.save(model.state_dict(), "best_absa_model.bin")
        print("--> Model saved!")

Epoch 1 | Train Loss: 846.4801 | Test Loss: 400.1598
--> Model saved!
Epoch 2 | Train Loss: 413.9245 | Test Loss: 366.2576
--> Model saved!
Epoch 3 | Train Loss: 375.2055 | Test Loss: 336.4491
--> Model saved!
Epoch 4 | Train Loss: 353.5832 | Test Loss: 322.1122
--> Model saved!
Epoch 5 | Train Loss: 335.2563 | Test Loss: 308.8159
--> Model saved!
Epoch 6 | Train Loss: 322.3645 | Test Loss: 297.7263
--> Model saved!
Epoch 7 | Train Loss: 308.7355 | Test Loss: 294.2802
--> Model saved!
Epoch 8 | Train Loss: 297.0205 | Test Loss: 282.6819
--> Model saved!
Epoch 9 | Train Loss: 287.3543 | Test Loss: 276.6003
--> Model saved!
Epoch 10 | Train Loss: 278.8959 | Test Loss: 274.6505
--> Model saved!
Epoch 11 | Train Loss: 269.1652 | Test Loss: 267.1797
--> Model saved!
Epoch 12 | Train Loss: 263.6290 | Test Loss: 264.6735
--> Model saved!
Epoch 13 | Train Loss: 255.8593 | Test Loss: 261.8922
--> Model saved!
Epoch 14 | Train Loss: 250.1822 | Test Loss: 264.2200
Epoch 15 | Train Loss: 244.5817 

In [43]:
def evaluate_model(model, data_loader, idx2tag):
    model.eval()
    y_true = []
    y_pred = []

    with torch.no_grad():
        for tokens, labels, masks in data_loader:
            tokens, labels, masks = tokens.to(device), labels.to(device), masks.to(device)

            paths = model(tokens, mask=masks)

            for i in range(len(paths)):
                true_tags = [idx2tag[l.item()] for l, m in zip(labels[i], masks[i]) if m == 1]
                pred_tags = [idx2tag[p] for p in paths[i]]

                pred_tags = pred_tags[:len(true_tags)]

                y_true.append(true_tags)
                y_pred.append(pred_tags)

    print(classification_report(y_true, y_pred))
    return f1_score(y_true, y_pred)

In [52]:
model.load_state_dict(torch.load("best_absa_model.bin"))
idx2tag = {v: k for k, v in tag2idx.items()}
f1 = evaluate_model(model, test_loader, idx2tag)

              precision    recall  f1-score   support

         NEG       0.64      0.22      0.33       983
         NEU       0.00      0.00      0.00        22
         POS       0.61      0.18      0.28      1369

   micro avg       0.62      0.20      0.30      2374
   macro avg       0.42      0.13      0.20      2374
weighted avg       0.62      0.20      0.30      2374



In [63]:
def predict_raw(text):
    model.eval()
    tokens = text.lower().split()
    indices = [word2idx.get(t, word2idx["<UNK>"]) for t in tokens]
    sentence_tensor = torch.LongTensor([indices]).to(device)

    with torch.no_grad():
        path = model(sentence_tensor) # Viterbi decoding

    tags = [idx2tag[p] for p in path[0]]
    return list(zip(tokens, tags))

print(predict_raw("The battery life was superb and overall experience was fine"))

[('the', 'O'), ('battery', 'B-POS'), ('life', 'I-POS'), ('was', 'O'), ('superb', 'O'), ('and', 'O'), ('overall', 'O'), ('experience', 'O'), ('was', 'O'), ('fine', 'O')]
